# knot — 01: deploy

Build a spec → see the SQL knot generates → execute it → look at the
tables. Subsequent notebooks (02_ingest, 03_query, ...) cover the
runtime concerns.

In [1]:
import json
import uuid

import psycopg

from knot import Spec, types

In [2]:
spec = Spec(identifier_slot_name="canonical_id")

person = spec.add_class("Person")
person.slot("name", types.TEXT, required=True)
person.slot("birth_country", types.TEXT)

movie = spec.add_class("Movie")
movie.slot("title", types.TEXT, required=True)
movie.slot("year", types.INTEGER)
movie.slot("director", person)  # FK — pass the class

spec

Spec(identifier_slot_name='canonical_id', classes=[OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='name', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='birth_country', type=<Primitive.TEXT: 'text'>, identifier=False, required=False, description=None)], description=None), OntologyClass(name='Movie', kind=<ClassKind.CONCRETE: 'concrete'>, is_a=None, mixins=[], slots=[Slot(name='canonical_id', type=<Primitive.TEXT: 'text'>, identifier=True, required=True, description=None), Slot(name='title', type=<Primitive.TEXT: 'text'>, identifier=False, required=True, description=None), Slot(name='year', type=<Primitive.INTEGER: 'integer'>, identifier=False, required=False, description=None), Slot(name='director', type=ClassRef(target=OntologyClass(name='Person', kind=<ClassKind.CONCRETE: 

In [3]:
# Host plumbing — psycopg connection + a fresh per-run schema name
# so re-running the notebook never collides with prior runs. We don't
# create the schema here: init_sql emits ``CREATE SCHEMA IF NOT EXISTS``
# as its first statement.
pg = psycopg.connect(
    host="localhost",
    port=5433,
    user="knot",
    password="knot",
    dbname="knot",
    autocommit=True,
)
schema = f"knot_play_{uuid.uuid4().hex[:8]}"
schema

'knot_play_8f2df8bd'

In [4]:
# Visualize the SQL knot would run against an empty schema. Nothing
# executes yet — just the script. Notice the order: schema → weight
# table → canonical tables → bindings tables → indexes → FK alters
# → resolved views → all-sources views → weight seed inserts.
print(spec.init_sql(schema=schema))

CREATE SCHEMA IF NOT EXISTS knot_play_8f2df8bd;

CREATE TABLE IF NOT EXISTS knot_play_8f2df8bd.source_weight (
    source_name text NOT NULL,
    class_name  text NOT NULL,
    slot_name   text NOT NULL,
    weight      double precision NOT NULL,
    PRIMARY KEY (source_name, class_name, slot_name)
);

CREATE TABLE IF NOT EXISTS knot_play_8f2df8bd.person (
    canonical_id text NOT NULL,
    name text NOT NULL,
    birth_country text,
    PRIMARY KEY (canonical_id)
);

CREATE TABLE IF NOT EXISTS knot_play_8f2df8bd.person_bindings (
    source_name text NOT NULL,
    source_identifier text NOT NULL,
    canonical_id text,
    name text,
    birth_country text,
    raw_payload jsonb NOT NULL DEFAULT '{}'::jsonb,
    er_metadata jsonb NOT NULL DEFAULT '{}'::jsonb,
    valid_from timestamptz NOT NULL DEFAULT now(),
    valid_to timestamptz,
    PRIMARY KEY (source_name, source_identifier, valid_from)
);

CREATE INDEX IF NOT EXISTS person_bindings_current_idx
    ON knot_play_8f2df8bd.perso

In [5]:
# Run the script. ``pg.execute`` accepts a multi-statement string;
# the autocommit connection commits each statement as it runs.
pg.execute(spec.init_sql(schema=schema))

<psycopg.Cursor [COMMAND_OK] [IDLE] (host=localhost port=5433 database=knot) at 0x71fdac184e90>

In [6]:
# What landed? Ask postgres directly via information_schema —
# describe-style introspection, not a knot read. For Person + Movie
# we expect:
#   - 1 invariant table: source_weight
#   - 2 canonical tables: person, movie
#   - 2 bindings tables: person_bindings, movie_bindings
#   - 2 resolved views: person_resolved, movie_resolved
#   - 2 all-sources views: person_all_sources, movie_all_sources
with pg.cursor() as cur:
    cur.execute(
        """
        SELECT table_name, table_type
        FROM information_schema.tables
        WHERE table_schema = %s
        ORDER BY table_type, table_name
        """,
        (schema,),
    )
    rows = cur.fetchall()

for name, kind in rows:
    print(f"{kind:11s}  {name}")

BASE TABLE   movie
BASE TABLE   movie_bindings
BASE TABLE   person
BASE TABLE   person_bindings
BASE TABLE   source_weight
VIEW         movie_all_sources
VIEW         movie_resolved
VIEW         person_all_sources
VIEW         person_resolved
